In [60]:
import qutip as qt
import numpy as np
import plotly.graph_objects as go



# Q3 Dual Rail Encoding
Here we're being asked to simulate a dual-rail encoding of a qubit by assigning a 2 level system to each of the photonic rails/modes. For the first part of the question we want to simulate the effects of a phase shifter on the 2nd rail (i.e. we're simulating a sigma_z gate of the dual-rail qubit). We also have loss rates for each rail kappa1 and kappa2 and we neglect dephasing.

We use the hamiltonian given in the notes for a phase shifter $$H_{PSk} =- \hbar*A_\phi*a^\dagger_k a_k$$

The master equation is the given by $$\dot{\rho} = -\frac{i}{\hbar}[H,\rho] + \kappa_1*D_1[a_1]\rho + \kappa_2*D_2[a_2]\rho$$

where our Lindblad operators are given by $$D_k[a_k]\rho = a_k \rho a_k^\dagger - \frac{1}{2}a_k^\dagger a_k \rho - \frac{1}{2}\rho a_k^\dagger a_k$$

and we use the annihilation operators as our jump operators since this corresponds to losing a single photon. 

In [61]:

a1 = qt.tensor(qt.destroy(2), qt.qeye(2))
a2 = qt.tensor(qt.qeye(2), qt.destroy(2))
n1 = qt.tensor(qt.num(2), qt.qeye(2))
n2 = qt.tensor(qt.qeye(2), qt.num(2))

def dual_rail_evo_phaseshift(rho_in, tfin, kappa1, kappa2, Aphi = np.pi/5):
    #returns the time evolution of the input state rho_in under the master equation given in the question for a time tfin.
    times = np.linspace(0, tfin, 100)
    H = -Aphi * n2
    c_ops = [np.sqrt(kappa1) * a1, np.sqrt(kappa2) * a2]
    evolution = qt.mesolve(H, rho_in, times, c_ops)
    return evolution

def plot_fidelities(evolution:list, psi, kappa1:list, kappa2:list, psi_in = "Plus State"):
    #plots the fidelity of the state at each time step in the evolution with respect to the target state psi.
    fig = go.Figure()

    for i, evo in enumerate(evolution):
        fidelities = [qt.fidelity(state, psi)**2 for state in evo.states] ## we take the square since qutip returns the square root of the fidelity, not the actual fidelity!
        fig.add_trace(go.Scatter(x=evo.times,y=fidelities,mode="lines+markers",name=f"κ₁={kappa1[i]}, κ₂={kappa2[i]}",))

    fig.update_layout(title=f"Fidelity vs Time ({psi_in})",xaxis_title="Time",yaxis_title="Fidelity",)
    fig.show()


vac = qt.tensor(qt.basis(2,0), qt.basis(2,0))
zero = qt.tensor(qt.basis(2,1), qt.basis(2,0))
one = qt.tensor(qt.basis(2,0), qt.basis(2,1))
psi_plus = (zero + one).unit()


evolutions = []
kappa1_list = [0.0, 0.2, 0.2]
kappa2_list = [0.0, 0.2, 1]

for i in range(3):
    evo = dual_rail_evo_phaseshift(psi_plus, 10, kappa1_list[i], kappa2_list[i])
    evolutions.append(evo)

plot_fidelities(evolutions, psi_plus, kappa1_list, kappa2_list)




These results are pretty much in line with what I would expect. For the case with no loss, we see that the fidelity oscillates between 1 and 0 as the state evolves under the phase shifter hamiltonian. This makes sense since our phase shifter is essentially a sigma_z gate on the dual-rail qubit, so we expect to see oscillations in the fidelity as the state evolves when we have no loss. 

For the case where we have equal loss on both, we expect to see a decay in the fidelity but to still see some oscillation our loss rate is low, the reason we see decay is because our state now slowly moves towards a vacuum state on top of the rotational dynamics. The vacuum state is perpendicular to both |0> and |1> so the fidelity inevitebly drops. We still see some oscillatory behaviour however because our loss rate is the same across both rails so the relative amplitudes of the oscillations are preserved. 

For the case where we have high loss on rail 2 and low loss on rail 1, we see a much more rapid decay in the fidelity which corresponds to loosing essentially all our photons in rail 2 at the begining. We then see a lower decay rate set by the loss of rail 1 which is much lower than the loss of rail 2, so we see a much slower decay in the fidelity after the initial rapid decay. We don't see oscillations because our phase shifter only acts on rail 2 and so when we lose all our photons in rail 2 we essentially just have amplitude damping on our state due to photon loss on rail 1 and our state moving closer to a vacuum state. 

# c) 
Here we simulate for the input state |10> which corresponds to the first rail having a single photon and the second rail having no photons.

In [71]:

# state |11>
psi_in = qt.tensor(qt.basis(2,1), qt.basis(2,1)).unit()
evolutions = []
for i in range(3):
    evo = dual_rail_evo_phaseshift(psi_in, 10, kappa1_list[i], kappa2_list[i])
    evolutions.append(evo)

plot_fidelities(evolutions, psi_plus, kappa1_list, kappa2_list, psi_in="|11> State")

Here again, the results are pretty straight forward to interpret. Since we start in a state perpendicular to the state we measure our fidelity against, we start with a fidelity of 0 in all cases. 

In the case of no loss, we will always stay in this state since our phase shifter acts on this state by evolving a phase on |11>. So we stay perpendicular to |0> and |1> and therefore perpendicular to our plus state. 

On the other hand, if we include loss terms, then our |11> state will evolve into some mix of |01> and |10> 

Now we want to simulate the effects of a beamsplitter on the dual-rail qubit. The hamiltonian for a beamsplitter is given by $$H_{BS} = -i\hbar*A_{\theta}(a_1^\dagger a_2 - a_1 a_2^\dagger)$$ for $\phi=0$

Our master equation is pretty much the same as before with our loss terms still being given by the same Lindblad operators.

That is our master equation is given by $$\dot{\rho} = -\frac{i}{\hbar}[H,\rho] + \kappa_1*D_1[a_1]\rho + \kappa_2*D_2[a_2]\rho$$

In [63]:
def dual_rail_evo_beamsplitter(rho_in, tfin, kappa1, kappa2, Atheta = np.pi/10):
    #returns the time evolution of the input state rho_in under the master equation given in the question for a time tfin.
    times = np.linspace(0, tfin, 100)
    H = -1j * Atheta * (a1.dag() * a2 - a1 * a2.dag())
    c_ops = [np.sqrt(kappa1) * a1, np.sqrt(kappa2) * a2]
    evolution = qt.mesolve(H, rho_in, times, c_ops)
    return evolution

evolutions = []
psi_zero = qt.tensor(qt.basis(2,1), qt.basis(2,0))
for i in range(3):
    evo = dual_rail_evo_beamsplitter(psi_zero, 10, kappa1_list[i], kappa2_list[i])
    evolutions.append(evo)
plot_fidelities(evolutions, psi_zero, kappa1_list, kappa2_list, psi_in="Zero State with Beamsplitter")

In [64]:
psi_in = qt.tensor(qt.basis(2,1), qt.basis(2,1)).unit()
evolutions = []
for i in range(3):
    evo = dual_rail_evo_beamsplitter(psi_in, 10, kappa1_list[i], kappa2_list[i])
    evolutions.append(evo)
plot_fidelities(evolutions, psi_zero, kappa1_list, kappa2_list, psi_in="|11> State with Beamsplitter")

A 50/50 beamsplitter corresponds to $\theta = \pi/4$, so we achieve a 50/50 beamsplitter at time t such that $A_{\theta}*t = \pi/4$.